
# Zero-shot Benchmark — **CLIP ViT-L/14@336** (open_clip)

Variante del notebook zero-shot coerente con la pipeline precedente, ma usando **ViT-L/14@336**.
Compatibile con **Colab + Google Drive**. Registra metriche e salva il CSV dei risultati.

**Metriche e visualizzazioni incluse**
- Top-1, Macro-F1, Top-5, Balanced Accuracy
- Classification report per classe (con `zero_division=0`)
- Confusion Matrix normalizzata
- Latency (ms/img) in inference
- GPU-h (0.0 per zero-shot), Effort umano (h), Note


In [ ]:

# === Install ===
!pip -q install open_clip_torch scikit-learn matplotlib pandas tqdm


In [ ]:

# === Imports & Config ===
import os, time, math
from datetime import datetime
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
import open_clip
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from sklearn.metrics import (accuracy_score, f1_score, top_k_accuracy_score,
                             balanced_accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# --------- CONFIG (MODIFICA QUI) ---------
DATA_ROOT = "/content/drive/MyDrive/ArchiStyles-2"  # path dataset su Drive
SPLIT = "test"              # "test" | "valid" | "train"
BATCH_SIZE = 32             # 336px usa più VRAM: riduci se serve
MODEL_NAME = "ViT-L-14-336"
PRETRAINED = "openai"

TEMPLATES = [
    "a photo of a {} building",
    "a photograph of {} architecture",
    "an example of {} architecture",
    "a building in {} style",
]

TEMP_GRID = [0.5, 1.0, 2.0]  # temperature search semplice

EFFORT_HOURS = 0.5
NOTES = "zero-shot L/14@336, prompt ensemble + temp search"
# -----------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


In [ ]:

# === Model & Dataset ===
model, _, preprocess = open_clip.create_model_and_transforms(
    model_name=MODEL_NAME,
    pretrained=PRETRAINED
)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.to(device).eval()

assert os.path.isdir(DATA_ROOT), f"DATA_ROOT non trovato: {DATA_ROOT}"
split_path = os.path.join(DATA_ROOT, SPLIT)
assert os.path.isdir(split_path), f"Split '{SPLIT}' non trovato in {DATA_ROOT}"

ds = ImageFolder(split_path, transform=preprocess)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
classnames = ds.classes
num_classes = len(classnames)
print("Classi:", num_classes)


In [ ]:

# === Prompt encode ===
def make_prompts(cname):
    return [t.format(cname) for t in TEMPLATES] + [f"{cname} architecture"]

all_prompts, class_offsets = [], [0]
for cname in classnames:
    ps = make_prompts(cname)
    all_prompts.extend(ps)
    class_offsets.append(len(all_prompts))

tokens = tokenizer(all_prompts).to(device)
with torch.no_grad():
    text_features = model.encode_text(tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)


In [ ]:

# === Eval helpers ===
def evaluate_with_temp(temp):
    y_true, y_pred = [], []
    probs_collector = []
    model_dtype = next(model.parameters()).dtype

    with torch.no_grad():
        for imgs, labels in dl:
            imgs = imgs.to(device, dtype=model_dtype)
            img_features = model.encode_image(imgs)
            img_features = img_features / img_features.norm(dim=-1, keepdim=True)

            logit_scale = model.logit_scale.exp() * temp
            logits = logit_scale * (img_features @ text_features.T)

            # aggregate prompts per class
            cls_logits = []
            for i in range(num_classes):
                s, e = class_offsets[i], class_offsets[i+1]
                cls_logits.append(logits[:, s:e].mean(dim=1))
            cls_logits = torch.stack(cls_logits, dim=1)

            prob = cls_logits.softmax(dim=-1)
            pred = prob.argmax(dim=-1)

            y_true.extend(labels.tolist())
            y_pred.extend(pred.tolist())
            probs_collector.append(prob.cpu().numpy())

    y_true = np.array(y_true, dtype=int)
    y_pred = np.array(y_pred, dtype=int)
    probs = np.concatenate(probs_collector, axis=0)

    top1 = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    top5 = top_k_accuracy_score(y_true, probs, k=min(5, num_classes), labels=np.arange(num_classes))
    bal_acc = balanced_accuracy_score(y_true, y_pred)

    return top1, macro_f1, top5, bal_acc, y_true, y_pred, probs


In [ ]:

# === Temperature search (nota: per rigore, farlo su VALID) ===
best = None
for t in TEMP_GRID:
    top1, macro_f1, top5, bal_acc, *_ = evaluate_with_temp(t)
    print(f"temp={t:.2f} -> top1={top1:.4f}  macroF1={macro_f1:.4f}  top5={top5:.4f}  balAcc={bal_acc:.4f}")
    if best is None or top1 > best[0]:
        best = (top1, macro_f1, top5, bal_acc, t)

best_top1, best_macro_f1, best_top5, best_balacc, best_temp = best
print("\nBest temperature:", best_temp)


In [ ]:

# === Final eval with best temp + latency + visuals ===
top1, macro_f1, top5, bal_acc, y_true, y_pred, probs = evaluate_with_temp(best_temp)

# Classification report
print("Accuracy:", f"{top1:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=classnames, digits=4, zero_division=0))

# Top-k detailed
for k in [1,2,3,5]:
    k = min(k, num_classes)
    tk = top_k_accuracy_score(y_true, probs, k=k, labels=np.arange(num_classes))
    print(f"Top-{k} Accuracy: {tk:.4f}")

# Confusion matrix (normalized)
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=classnames,
    normalize="true", cmap="Blues", include_values=False, xticks_rotation=90, ax=ax
)
ax.set_title("Confusion Matrix (normalized)")
plt.tight_layout()
plt.show()

# Latency measurement
def measure_latency(num_batches=20):
    model_dtype = next(model.parameters()).dtype
    # warmup
    it = iter(dl)
    for _ in range(3):
        try: imgs, _ = next(it)
        except StopIteration:
            it = iter(dl); imgs, _ = next(it)
        imgs = imgs.to(device, dtype=model_dtype)
        with torch.no_grad(): _ = model.encode_image(imgs)
    if torch.cuda.is_available(): torch.cuda.synchronize()

    times = []
    it = iter(dl)
    for _ in range(min(num_batches, max(1, math.ceil(len(ds)/BATCH_SIZE)))):
        try: imgs, _ = next(it)
        except StopIteration:
            it = iter(dl); imgs, _ = next(it)
        imgs = imgs.to(device, dtype=model_dtype)
        t0 = time.time()
        with torch.no_grad(): _ = model.encode_image(imgs)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        dt = time.time() - t0
        times.append((dt / imgs.size(0)) * 1000.0)
    return float(np.mean(times))

latency_ms = measure_latency()

gpu_hours = 0.0  # zero-shot

print(f"FINAL -> Top-1={top1:.4f} | Macro-F1={macro_f1:.4f} | Top-5={top5:.4f} | BalAcc={bal_acc:.4f}")
print(f"Latency: {latency_ms:.1f} ms/img | GPU-h: {gpu_hours:.2f}")


In [ ]:

# === Save results to CSV ===
row = {
    "Timestamp": datetime.utcnow().isoformat(),
    "Model": MODEL_NAME,
    "Pretrained": PRETRAINED,
    "Scenario": f"Zero-shot ({SPLIT})",
    "Techniques": "Prompt ensemble + temperature search",
    "Top-1 Acc": round(float(top1), 4),
    "Macro-F1": round(float(macro_f1), 4),
    "Top-5 Acc": round(float(top5), 4),
    "Balanced Acc": round(float(bal_acc), 4),
    "GPU-h": gpu_hours,
    "Latency (ms/img)": round(float(latency_ms), 1),
    "Effort umano (h)": float(EFFORT_HOURS),
    "Note": NOTES,
    "Classes": int(num_classes),
    "Batch size": int(BATCH_SIZE)
}
import pandas as pd
df = pd.DataFrame([row])
display(df)

local_csv = "/content/benchmark_results.csv"
if os.path.exists(local_csv):
    prev = pd.read_csv(local_csv)
    out = pd.concat([prev, df], ignore_index=True)
else:
    out = df.copy()
out.to_csv(local_csv, index=False)

drive_csv = os.path.join(DATA_ROOT, "benchmark_results.csv")
try:
    if os.path.exists(drive_csv):
        prev_d = pd.read_csv(drive_csv)
        out_d = pd.concat([prev_d, df], ignore_index=True)
    else:
        out_d = df.copy()
    out_d.to_csv(drive_csv, index=False)
    print("Saved CSV to:", local_csv, "and", drive_csv)
except Exception as e:
    print("Saved CSV to:", local_csv)
    print("Drive save failed:", e)
